# raw_to_dst — Convert raw tag CSV to standard format

Run this **once** per tag. Produces:
```
{TAG_ROOT}/{tag_name}/
    dst.csv            ← time, temperature, pressure, light
    tagging_event.csv  ← release + fish_death
    metadata.json      ← tag_name, tag_type
```
After this, use `load_tag(tag_root=TAG_ROOT, tag_name=tag_name)` normally.


In [ ]:
import sys

sys.path.append("../")


TAG_ROOT = "/tmp/tags"

TAG_TYPE = "wc_psat"
tag_name = "263936"
RAW_DATA_PATH = (
    "../docs/specs/reference/tuna-tristan/tuna-tristan/263936/263936-Series.csv"
)
TAGGING_EVENTS_PATH = "263936/tagging_events.csv"
TAG_ROOT = ""

scratch_root = "s3://gfts-ifremer/tuna_mediterranean/tags/formatted"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

In [ ]:
from pangeo_fish.light.ingest import prepare_tag_folder

prepare_tag_folder(
    RAW_DATA_PATH,
    TAG_TYPE,
    tagging_events_path=TAGGING_EVENTS_PATH,
    output_dir=TAG_ROOT,
    tag_name=tag_name,
)

## Verify

In [ ]:
import sys

sys.path.append("../")
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(tag_root=TAG_ROOT, tag_name=tag_name)
print(f"Deployment: {time_slice.start} → {time_slice.stop}")
print(f"tag_log: {len(tag_log.time):,} timesteps")
tag

In [ ]:
import s3fs

s3 = s3fs.S3FileSystem(**storage_options)
local_folder = f"{TAG_ROOT}/{tag_name}"
s3_folder = f"{scratch_root}/{tag_name}"

for fname in ["dst.csv", "tagging_event.csv", "metadata.json"]:
    local = f"{local_folder}/{fname}"
    remote = f"{s3_folder}/{fname}"
    s3.put(local, remote)
    print(f"Uploaded → {remote}")

print(f"\nDone. Load with:")
print(f'  load_tag(tag_root="{scratch_root}", tag_name="{tag_name}")')